# MaskOCR for Handwriting recognition

**Implement <a href='https://arxiv.org/pdf/2206.00311.pdf'>MaskOCR paper</a>**<br>
**Description:** Implement MaskOCR model with 3 training phases. <br>
**Implementer:** Lam Thanh Tin - 3733383

## Install dependencies

In [ ]:
! pip install --quiet timm einops tensorboard

## Data collections
Run 1 time to download dataset.

In [ ]:
!wget -q https://github.com/sayakpaul/Handwriting-Recognizer-in-Keras/releases/download/v1.0.0/IAM_Words.zip
!unzip -qq IAM_Words.zip
!mkdir data data/words data/vi
!tar -xf IAM_Words/words.tgz -C data/words
!mv IAM_Words/words.txt data

!gdown --folder 1vyw6L5fLcbrT7uBqonbdkkB_Ci14ph6j
!unzip -qq /content/Kukdan/training_data.zip
!unzip -qq /content/Kukdan/public_test_data.zip
!mv /content/new_train /content/data/vi
!mv /content/new_public_test /content/data/vi
!mv /content/Kukdan/train_gt.txt /content/data/vi

## Data loader helpers
Function to load and create masks on IAM and Vietnamese handwriting datasets.

In [ ]:
import os

from datetime import datetime
from torch.utils.tensorboard import SummaryWriter
from tqdm import tqdm
from torch.utils.data.dataset import Dataset
from torch.utils.data import DataLoader, random_split
from typing import Optional, Callable
import multiprocessing
import torchvision.transforms as T
from PIL import Image
import urllib.request
from urllib.error import HTTPError

import numpy as np
import random
# import lightning as L
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
import torch.utils.data as data
from einops import repeat
import torchvision
# from lightning.pytorch.callbacks import LearningRateMonitor, ModelCheckpoint
from torchvision import transforms

IMAGENET_DEFAULT_MEAN = [0.485, 0.456, 0.406]
IMAGENET_DEFAULT_STD = [0.229, 0.224, 0.225]
IMAGENET_INCEPTION_MEAN = [0.5, 0.5, 0.5]
IMAGENET_INCEPTION_STD = [0.5, 0.5, 0.5]
EXTENSIONS = ['jpg', 'png', 'jpeg']

def seed_worker(worker_id):
    worker_seed = torch.initial_seed() % 2**32
    np.random.seed(worker_seed)
    random.seed(worker_seed)

class Obj():
    def __init__(self, data):
        self._data = data

    def __getattr__(self, key):
        if key in self._data:
            if isinstance(self._data[key], dict):
                return Obj(self._data[key])
            else:
                return self._data[key]
        else:
            raise AttributeError(f"'Obj' object has no attribute '{key}'")

    def __repr__(self):
        return repr(self._data)

    def __setattr__(self, key, value):
        if "_data" in self.__dict__ and key in self._data:
            if isinstance(self._data[key], dict):
                self._data[key] = value._data if isinstance(value, Obj) else value
            else:
                self._data[key] = value
        else:
            super().__setattr__(key, value)

class MaskGenerator():
    def __init__(self, input_size, ratio_masking_patches, for_decoder=False):
        if not isinstance(input_size, tuple):
            input_size = (input_size, ) * 2
        self.height, self.width = input_size
        self.for_decoder = for_decoder
        self.ratio_masking_patches = ratio_masking_patches

        # mask settings for decoder
        if self.for_decoder:
            self.num_patches = self.height * self.width
            self.num_masking_patches = int(ratio_masking_patches * self.num_patches)
        else:
            self.num_patches = self.height * self.width
            self.num_masking_patches = int(ratio_masking_patches * self.num_patches)

    def __repr__(self):
        repr_str = "Mask: total patches {}, mask patches {}".format(
            self.num_patches, self.num_masking_patches
        )
        return repr_str

    def check_valid_mask(self, mask_list): # check if mask list is valid (2 mask is not adjacent)
        if len(mask_list) == 1: return True

        for i in range(1, len(mask_list)):
            if mask_list[i] == 1 and mask_list[i-1] == mask_list[i]:
              return False
        return True


    def __call__(self, label=None):
        if self.for_decoder: # if generating mask for decoder
            label_len = len(label)
            patches2base_ratio = int(self.num_patches/label_len)
            num_masking_patches = int(self.ratio_masking_patches * label_len)

            # Define parameters

            mask = np.hstack([
                            np.ones(num_masking_patches), # 1 is mask
                            np.zeros(label_len - num_masking_patches), # 0 is visible
                        ])
            while(not self.check_valid_mask(mask)):
                np.random.shuffle(mask)

            mask = np.append(mask, 0.)

            mask_list_extend = []
            for i in range(len(mask)):
                for j in range(patches2base_ratio if i < len(mask) - 1 else self.num_patches - (len(mask) - 1) * patches2base_ratio):
                    mask_list_extend.append(mask[i])
            mask_list_extend = np.array(mask_list_extend)

            # ensure that mask positions in mask_list_extend are equal to masking patches
            base_sum = sum(mask_list_extend)
            if self.num_masking_patches > sum(mask_list_extend):
                for i in range(int(self.num_masking_patches - sum(mask_list_extend))):
                    indices = np.where(mask_list_extend == 0.)[0]
                    mask_list_extend[np.random.choice(indices)] = 1.
            if self.num_masking_patches < sum(mask_list_extend):
                for i in range(int(sum(mask_list_extend) - self.num_masking_patches)):
                    indices = np.where(mask_list_extend == 1.)[0]
                    mask_list_extend[np.random.choice(indices)] = 0.

            # fix error for batching different shape of elements
            mask_for_batch = []
            for i in range(len(mask)):
                count = 0
                for j in range(patches2base_ratio if i < len(mask) - 1 else self.num_patches - (len(mask) - 1) * patches2base_ratio):
                    if count == 0:
                        mask_for_batch.append(mask[i])
                        count += 1
                    else: mask_for_batch.append(999) # 999 is a special token adding just to fix the error, can be remove after loaded to the model.
            mask_for_batch = np.array(mask_for_batch)

            return mask_for_batch, mask_list_extend

        else:
            mask = np.hstack([
                np.zeros(self.num_patches - self.num_masking_patches),
                np.ones(self.num_masking_patches),
            ])
            np.random.shuffle(mask)

        return mask

class OCR_Dataset(Dataset):
    def __init__(self, args):
        self.image_paths = []

        self.labels = []
        self.vocab = set()
        self.char_to_index = None
        self.index_to_char = None
        self.args = args
        self.max_text_length  = 0

        self.mode = args.mode
        self.name = args.dataset.name # ex structure: ['iam', 'vi']
        window_size = (args.input_size // args.patch_size, args.input_size // args.patch_size)

        imagenet_default_mean_and_std = args.imagenet_default_mean_and_std
        mean = IMAGENET_INCEPTION_MEAN if not imagenet_default_mean_and_std else IMAGENET_DEFAULT_MEAN
        std = IMAGENET_INCEPTION_STD if not imagenet_default_mean_and_std else IMAGENET_DEFAULT_STD

        if 'iam' in self.name:
            # read images labels for IAMDataset
            words = open(f"{args.dataset.iam_path}/words.txt", "r").readlines()

            for line in range(len(words)):
                if words[line][0] == '#':
                    continue
                if words[line].split(" ")[1] != 'err' and (self.mode != 'd_pretrain' or self.mode == 'd_pretrain' and len(words[line].split(' ')[-1].split('\n')[0]) >= 4):  # We don't need to deal with errored entries (for decoder pretrain: and words that are too short (less than 3 character)).
                    # load img
                    folder_paths = words[line].split(' ')[0].split('-') # ex: a01-000u-00-00
                    img_path = f'{args.dataset.iam_path}/words/{folder_paths[0]}/{"-".join(folder_paths[:2])}/{"-".join(folder_paths)}.png'
                    try:
                        img = Image.open(img_path)
                        img.verify()
                        img.close()
                        self.image_paths.append(img_path)
                    except:
                        continue

                    # load label
                    label = words[line].split(' ')[-1].split('\n')[0]
                    self.labels.append(label)
                    self.vocab.update(list(label))
                    self.max_text_length = max(self.max_text_length, len(label))
                else:
                    continue
        if 'vi' in self.name:
            # read images labels for Vietnamese handwriting dataset
            words = open(f"{args.dataset.vi_path}/train_gt.txt", "r", encoding="utf-8").readlines()

            for line in range(len(words)):
                label_tuple = words[line].rstrip('\n').split('\t') # strip the newline so it is not part of the label
                if len(label_tuple) < 2 or label_tuple[-1] == '':
                    continue
                if (self.mode != 'd_pretrain') or (self.mode == 'd_pretrain' and len(label_tuple[-1]) >= 4):
                    # load img
                    img_path = f'{args.dataset.vi_path}/new_train/{label_tuple[0]}'
                    try:
                        img = Image.open(img_path)
                        img.verify()
                        img.close()
                        self.image_paths.append(img_path)
                    except:
                        continue

                    # load label
                    self.labels.append(label_tuple[-1])
                    self.vocab.update(label_tuple[-1])
                    self.max_text_length = max(self.max_text_length, len(label_tuple[-1]))
                else: continue



        self.vocab = ''.join(sorted(self.vocab)) + 'Ë' # EOS token: Ë
        self.padding_token = len(self.vocab) - 1 # EOS token
        self.char_to_index = {char: index for index, char in enumerate(self.vocab)}
        self.char_to_index['Ë'] = len(self.char_to_index) + 1
        self.index_to_char = [char for char in self.vocab]
        # self.index_to_char.append('Ë')

        # Transformations
        if args.color_jitter > 0:
            self.common_transform = T.Compose([
                T.ColorJitter(args.color_jitter, args.color_jitter, args.color_jitter),
                T.RandomResizedCrop((224, 224), scale=(0.8, 1.0), ratio=(0.9, 1.1))
            ])
        else:
            self.common_transform = T.Compose([
                T.RandomResizedCrop((224, 224), scale=(0.8, 1.0), ratio=(0.9, 1.1))
            ])

        self.patch_transform = T.Compose([
            T.ToTensor(),
            T.Normalize(
                mean=torch.tensor(mean),
                std=torch.tensor(std))
        ])

        self.visual_token_transform = T.Compose([
            T.ToTensor(),
            T.Normalize(
                mean=IMAGENET_INCEPTION_MEAN,
                std=IMAGENET_INCEPTION_STD,
            )
        ])

        # Mask generator
        if self.mode == 'd_pretrain':
            self.masked_position_generator = MaskGenerator(window_size, ratio_masking_patches=args.ratio_mask_patches, for_decoder=True)
        else:
            self.masked_position_generator = MaskGenerator(window_size, ratio_masking_patches=args.ratio_mask_patches)


    def vectorize_label(self, label, text_masked_pos=None, status=0):
        length = len(label)
        encoded_label = torch.tensor([self.char_to_index[char] for char in label])
        # mask label for decoder pretraining: just remain masked label to calculate the loss for the decoder to learn the representation
        # 1 is masked and 0 is visible
        if text_masked_pos is not None:
            encoded_label = torch.tensor([encoded_label[i] if text_masked_pos[i] == status else self.padding_token for i in range(len(encoded_label))])
        pad_amount = self.max_text_length - length - 1
        padded_tensor = torch.zeros(self.max_text_length, dtype=torch.int64) + self.padding_token
        padded_tensor[:encoded_label.size(0)] = encoded_label

        return padded_tensor

    def decode_label(self, vector, pred=True):
        # label = ''.join([self.index_to_char[num] for num in vector[vector != self.padding_token]])
        # return label
        label = ''
        for num in torch.as_tensor(vector).round().long().tolist():
            if pred == False:
                if num == self.padding_token:
                    continue
            else:
                if num == self.padding_token:
                    break
            label += self.index_to_char[num]
        return label

    def __len__(self):
        return len(self.image_paths)


    def __getitem__(self, idx):
        img_path = self.image_paths[idx]
        image = Image.open(img_path).convert('RGB')

        # Apply transformations
        image = self.common_transform(image)

        if self.mode.lower() == 'd_pretrain':
            mask_idx, mask_idx_extend = self.masked_position_generator(self.labels[idx]) # set mask index to True to take text characters at masked positions as label

            return self.patch_transform(image), \
                   mask_idx, \
                   mask_idx_extend, \
                   self.vectorize_label(self.labels[idx], mask_idx[mask_idx!=999]), \
                   self.vectorize_label(self.labels[idx], mask_idx[mask_idx!=999], status=1.), \
                   torch.tensor([int(len(self.labels[idx]) * self.args.ratio_mask_patches)])

        elif self.mode.lower() == 'e_pretrain':
              return self.patch_transform(image), \
                     self.visual_token_transform(image), \
                     self.masked_position_generator()

        return self.patch_transform(image), \
               self.visual_token_transform(image), \
               self.masked_position_generator(), \
               self.vectorize_label(self.labels[idx]), \
               torch.tensor([len(self.labels[idx])])

        # if self.mode.lower() == 'pretrain':
        #     return self.patch_transform(image), \
        #           self.visual_token_transform(image), \
        #           self.masked_position_generator()
        # else: return self.patch_transform(image)

## Base Structures
Base structures and elements for encoder and decoder


In [ ]:
from functools import partial
import torch.nn.init as init
from timm.models.layers import drop_path, to_2tuple, trunc_normal_
import math


def logger(msg, type='INFO'):
    print(f'[{type}] {msg}')

def trunc_normal_(tensor, mean=0., std=1.):
    init.trunc_normal_(tensor, mean=mean, std=std, a=-std, b=std)

class Mlp(nn.Module):
    def __init__(self, in_features, hidden_features=None, out_features=None, act_layer=nn.GELU, drop=0.):
        super().__init__()
        out_features = out_features or in_features
        hidden_features = hidden_features or in_features
        self.fc1 = nn.Linear(in_features, hidden_features)
        self.act = act_layer()
        self.fc2 = nn.Linear(hidden_features, out_features)
        self.drop = nn.Dropout(drop)

    def forward(self, x):
        x = self.fc1(x)
        x = self.act(x)
        # x = self.drop(x)
        # commit this for the orignal BERT implement
        x = self.fc2(x)
        x = self.drop(x)
        return x

class PatchEmbed(nn.Module):
    """ Image to Patch Embedding
    """
    def __init__(self, img_size=224, patch_size=16, in_chans=3, embed_dim=768):
        super().__init__()
        img_size = to_2tuple(img_size)
        patch_size = to_2tuple(patch_size)
        num_patches = (img_size[1] // patch_size[1]) * (img_size[0] // patch_size[0])
        self.patch_shape = (img_size[0] // patch_size[0], img_size[1] // patch_size[1])
        self.img_size = img_size
        self.patch_size = patch_size
        self.num_patches = num_patches

        self.proj = nn.Conv2d(in_chans, embed_dim, kernel_size=patch_size, stride=patch_size)

    def forward(self, x, **kwargs):
        B, C, H, W = x.shape
        # FIXME look at relaxing size constraints
        assert H == self.img_size[0] and W == self.img_size[1], \
            f"Input image size ({H}*{W}) doesn't match model ({self.img_size[0]}*{self.img_size[1]})."
        x = self.proj(x).flatten(2).transpose(1, 2)
        return x

class DropPath(nn.Module):
    """Drop paths (Stochastic Depth) per sample  (when applied in main path of residual blocks).
    """
    def __init__(self, drop_prob=None):
        super(DropPath, self).__init__()
        self.drop_prob = drop_prob

    def forward(self, x):
        return drop_path(x, self.drop_prob, self.training)

    def extra_repr(self) -> str:
        return 'p={}'.format(self.drop_prob)

class Attention(nn.Module):
    def __init__(
            self, dim, num_heads=8, qkv_bias=False, qk_scale=None, attn_drop=0.,
            proj_drop=0., window_size=None, attn_head_dim=None):
        super().__init__()
        self.num_heads = num_heads
        head_dim = dim // num_heads
        if attn_head_dim is not None:
            head_dim = attn_head_dim
        all_head_dim = head_dim * self.num_heads
        self.scale = qk_scale or head_dim ** -0.5

        self.qkv = nn.Linear(dim, all_head_dim * 3, bias=False)
        if qkv_bias:
            self.q_bias = nn.Parameter(torch.zeros(all_head_dim))
            self.v_bias = nn.Parameter(torch.zeros(all_head_dim))
        else:
            self.q_bias = None
            self.v_bias = None

        self.attn_drop = nn.Dropout(attn_drop)
        self.proj = nn.Linear(all_head_dim, dim)
        self.proj_drop = nn.Dropout(proj_drop)

    def forward(self, x, bool_masked_pos=None):

        B, N, C = x.shape
        qkv_bias = None
        if self.q_bias is not None:
            qkv_bias = torch.cat((self.q_bias, torch.zeros_like(self.v_bias, requires_grad=False), self.v_bias))

        qkv = F.linear(input=x, weight=self.qkv.weight, bias=qkv_bias)
        qkv = qkv.reshape(B, N, 3, self.num_heads, -1).permute(2, 0, 3, 1, 4)
        q, k, v = qkv[0], qkv[1], qkv[2]

        q = q * self.scale
        attn = (q @ k.transpose(-2, -1))    # (B, N_head, N, N)

        attn = attn.softmax(dim=-1)
        attn = self.attn_drop(attn)

        x = (attn @ v).transpose(1, 2).reshape(B, N, -1)
        x = self.proj(x)
        x = self.proj_drop(x)

        return x

'''
Modified from Attention()
'''
class CrossAttention(nn.Module):
    def __init__(
            self, dim, num_heads=8, qkv_bias=False, qk_scale=None, attn_drop=0.,
            proj_drop=0., window_size=None, attn_head_dim=None):
        super().__init__()
        self.num_heads = num_heads
        head_dim = dim // num_heads
        if attn_head_dim is not None:
            head_dim = attn_head_dim
        all_head_dim = head_dim * self.num_heads
        self.scale = qk_scale or head_dim ** -0.5

        self.q = nn.Linear(dim, all_head_dim, bias=False)
        self.k = nn.Linear(dim, all_head_dim, bias=False)
        self.v = nn.Linear(dim, all_head_dim, bias=False)

        if qkv_bias:
            self.q_bias = nn.Parameter(torch.zeros(all_head_dim))
            self.v_bias = nn.Parameter(torch.zeros(all_head_dim))
        else:
            self.q_bias = None
            self.k_bias = None
            self.v_bias = None

        self.attn_drop = nn.Dropout(attn_drop)
        self.proj = nn.Linear(all_head_dim, dim)
        self.proj_drop = nn.Dropout(proj_drop)

    def forward(self, x, bool_masked_pos=None, k=None, v=None):
        B, N, C = x.shape
        N_k = k.shape[1]
        N_v = v.shape[1]

        q_bias, k_bias, v_bias = None, None, None
        if self.q_bias is not None:
            q_bias = self.q_bias
            k_bias = torch.zeros_like(self.v_bias, requires_grad=False)
            v_bias = self.v_bias

        q = F.linear(input=x, weight=self.q.weight, bias=q_bias)
        q = q.reshape(B, N, 1, self.num_heads, -1).permute(2, 0, 3, 1, 4).squeeze(0)    # (B, N_head, N_q, dim)

        k = F.linear(input=k, weight=self.k.weight, bias=k_bias)
        k = k.reshape(B, N_k, 1, self.num_heads, -1).permute(2, 0, 3, 1, 4).squeeze(0)

        v = F.linear(input=v, weight=self.v.weight, bias=v_bias)
        v = v.reshape(B, N_v, 1, self.num_heads, -1).permute(2, 0, 3, 1, 4).squeeze(0)

        q = q * self.scale
        attn = (q @ k.transpose(-2, -1))      # (B, N_head, N_q, N_k)

        attn = attn.softmax(dim=-1)
        attn = self.attn_drop(attn)

        x = (attn @ v).transpose(1, 2).reshape(B, N, -1)
        x = self.proj(x)
        x = self.proj_drop(x)

        return x


class Block(nn.Module):

    def __init__(self, dim, num_heads, mlp_ratio=4., qkv_bias=False, qk_scale=None, drop=0., attn_drop=0.,
                 drop_path=0., init_values=None, act_layer=nn.GELU, norm_layer=nn.LayerNorm,
                 window_size=None, attn_head_dim=None):
        super().__init__()
        self.norm1 = norm_layer(dim)
        self.attn = Attention(
            dim, num_heads=num_heads, qkv_bias=qkv_bias, qk_scale=qk_scale,
            attn_drop=attn_drop, proj_drop=drop, window_size=window_size, attn_head_dim=attn_head_dim)
        self.drop_path = DropPath(drop_path) if drop_path > 0. else nn.Identity()
        self.norm2 = norm_layer(dim)
        mlp_hidden_dim = int(dim * mlp_ratio)
        self.mlp = Mlp(in_features=dim, hidden_features=mlp_hidden_dim, act_layer=act_layer, drop=drop)

        if init_values > 0:
            self.gamma_1 = nn.Parameter(init_values * torch.ones((dim)),requires_grad=True)
            self.gamma_2 = nn.Parameter(init_values * torch.ones((dim)),requires_grad=True)
        else:
            self.gamma_1, self.gamma_2 = None, None

    def forward(self, x, bool_masked_pos=None):
        if self.gamma_1 is None:
            x = x + self.drop_path(self.attn(self.norm1(x), bool_masked_pos))
            x = x + self.drop_path(self.mlp(self.norm2(x)))
        else:
            x = x + self.drop_path(self.gamma_1 * self.attn(self.norm1(x), bool_masked_pos))
            x = x + self.drop_path(self.gamma_2 * self.mlp(self.norm2(x)))

        return x


class RegressorBlock(nn.Module):
    def __init__(self, dim, num_heads, mlp_ratio=4., qkv_bias=False, qk_scale=None, drop=0., attn_drop=0.,
                 drop_path=0., init_values=None, act_layer=nn.GELU, norm_layer=nn.LayerNorm,
                 window_size=None, attn_head_dim=None):
        super().__init__()
        self.norm1_q = norm_layer(dim)
        self.norm1_k = norm_layer(dim)
        self.norm1_v = norm_layer(dim)
        self.norm2_cross = norm_layer(dim)
        self.cross_attn =  CrossAttention(
            dim, num_heads=num_heads, qkv_bias=qkv_bias, qk_scale=qk_scale,
            attn_drop=attn_drop, proj_drop=drop, window_size=window_size, attn_head_dim=attn_head_dim)

        self.drop_path = DropPath(drop_path) if drop_path > 0. else nn.Identity()
        mlp_hidden_dim = int(dim * mlp_ratio)

        self.mlp_cross = Mlp(in_features=dim, hidden_features=mlp_hidden_dim, act_layer=act_layer, drop=drop)

        if init_values > 0:
            self.gamma_1_cross = nn.Parameter(init_values * torch.ones((dim)),requires_grad=True)
            self.gamma_2_cross = nn.Parameter(init_values * torch.ones((dim)),requires_grad=True)
        else:
            self.gamma_1_cross = nn.Parameter(torch.ones((dim)),requires_grad=False)
            self.gamma_2_cross = nn.Parameter(torch.ones((dim)),requires_grad=False)

    def forward(self, x_q, x_kv, pos_q, pos_k, bool_masked_pos):
        x = x_q + self.drop_path(self.gamma_1_cross * self.cross_attn(self.norm1_q(x_q + pos_q),
         bool_masked_pos, k=self.norm1_k(x_kv + pos_k), v=self.norm1_v(x_kv)))
        x = self.norm2_cross(x)
        x = x + self.drop_path(self.gamma_2_cross * self.mlp_cross(x))

        return x


'''
Encoder that extracts representations
'''
class VisionTransformerEncoder(nn.Module):
    def __init__(self, img_size=224, patch_size=16, in_chans=3, vocab_size=8192, embed_dim=768, depth=12,
                 num_heads=12, mlp_ratio=4., qkv_bias=True, qk_scale=None, drop_rate=0., attn_drop_rate=0.,
                 drop_path_rate=0., norm_layer=None, init_values=None, attn_head_dim=None,
                 use_abs_pos_emb=True, init_std=0.02, use_cls_token=True, args=None, **kwargs):
        super().__init__()
        self.args = args
        self.num_features = self.embed_dim = embed_dim

        self.patch_embed = PatchEmbed(
            img_size=img_size, patch_size=patch_size, in_chans=in_chans, embed_dim=embed_dim)
        self.num_patches = self.patch_embed.num_patches
        self.use_cls_token = use_cls_token

        # generate class token and pos embed
        self.cls_token = nn.Parameter(torch.zeros(1, 1, embed_dim))
        self.pos_embed = self.build_2d_sincos_position_embedding(embed_dim, use_cls_token=use_cls_token)
        self.pos_drop = nn.Dropout(p=drop_rate)

        dpr = [x.item() for x in torch.linspace(0, drop_path_rate, depth)]
        self.blocks = nn.ModuleList([
            Block(
                dim=embed_dim, num_heads=num_heads, mlp_ratio=mlp_ratio, qkv_bias=qkv_bias, qk_scale=qk_scale,
                drop=drop_rate, attn_drop=attn_drop_rate, drop_path=dpr[i], norm_layer=norm_layer,
                init_values=init_values, window_size=None,
                attn_head_dim=attn_head_dim,
            )
            for i in range(depth)])
        self.norm = norm_layer(embed_dim)

        self.init_std = init_std

        # init the model
        trunc_normal_(self.cls_token, std=self.init_std)
        self.apply(self._init_weights)
        # rescale init function from beit
        # if it is not activated, it will be overwritten
        self.fix_init_weight()

    def build_2d_sincos_position_embedding(self, embed_dim=768, temperature=10000., use_cls_token=False):
        h, w = self.patch_embed.patch_shape
        grid_w = torch.arange(w, dtype=torch.float32)
        grid_h = torch.arange(h, dtype=torch.float32)
        grid_w, grid_h = torch.meshgrid(grid_w, grid_h)
        assert embed_dim % 4 == 0, 'Embed dimension must be divisible by 4 for 2D sin-cos position embedding'
        pos_dim = embed_dim // 4
        omega = torch.arange(pos_dim, dtype=torch.float32) / pos_dim
        omega = 1. / (temperature ** omega)
        out_w = torch.einsum('m,d->md', [grid_w.flatten(), omega])
        out_h = torch.einsum('m,d->md', [grid_h.flatten(), omega])
        pos_emb = torch.cat([torch.sin(out_w), torch.cos(out_w), torch.sin(out_h), torch.cos(out_h)], dim=1)[None, :, :]

        if not use_cls_token:
            pos_embed = nn.Parameter(pos_emb)
        else:
            pe_token = torch.zeros([1, 1, embed_dim], dtype=torch.float32)
            pos_embed = nn.Parameter(torch.cat([pe_token, pos_emb], dim=1))
        pos_embed.requires_grad = False
        return pos_embed

    def fix_init_weight(self):
        def rescale(param, layer_id):
            param.div_(math.sqrt(2.0 * layer_id))

        for layer_id, layer in enumerate(self.blocks):
            rescale(layer.attn.proj.weight.data, layer_id + 1)
            rescale(layer.mlp.fc2.weight.data, layer_id + 1)

    def _init_weights(self, m):
        if isinstance(m, nn.Linear):
            trunc_normal_(m.weight, std=self.init_std)
            if isinstance(m, nn.Linear) and m.bias is not None:
                nn.init.constant_(m.bias, 0)
        elif isinstance(m, nn.LayerNorm):
            nn.init.constant_(m.bias, 0)
            nn.init.constant_(m.weight, 1.0)
        elif isinstance(m, nn.Conv2d):
            trunc_normal_(m.weight, std=self.init_std)
            if m.bias is not None:
                nn.init.constant_(m.bias, 0)

    @torch.jit.ignore
    def no_weight_decay(self):
        return {'pos_embed', 'cls_token'}

    def get_num_layers(self):
        return len(self.blocks)

    def forward_features(self, x, bool_masked_pos):
        x = self.patch_embed(x, bool_masked_pos=bool_masked_pos)
        batch_size, seq_len, dim = x.size()

        cls_tokens = self.cls_token.expand(batch_size, -1, -1)

        mask_pos = bool_masked_pos.bool()
        unmask_pos = torch.logical_not(mask_pos)

        # unmasked embeddings
        x_unmasked = x[unmask_pos].reshape(batch_size, -1, dim) # visible patches
        x_unmasked = torch.cat((cls_tokens, x_unmasked), dim=1)

        if self.pos_embed is not None:
            if self.use_cls_token:
                pos_embed = self.pos_embed.expand(batch_size, self.num_patches+1, dim)
                pos_embed_unmasked = pos_embed[:,1:][unmask_pos].reshape(batch_size, -1, dim)
                pos_embed_unmasked = torch.cat((pos_embed[:,:1], pos_embed_unmasked),dim=1)
            else: pos_embed = self.pos_embed[unmask_pos].reshape(batch_size, -1, dim)
            x_unmasked = x_unmasked + pos_embed_unmasked

        x_unmasked = self.pos_drop(x_unmasked)

        for blk in self.blocks:
            x_unmasked = blk(x_unmasked, bool_masked_pos)

        x_unmasked = self.norm(x_unmasked)

        return x_unmasked

    def forward(self, x, bool_masked_pos, return_all_tokens=False):
        x = self.forward_features(x, bool_masked_pos=bool_masked_pos)
        return x

'''
Latent context regressor + decoder that solves the pretext task.
'''
class VisionTransformerNeck(nn.Module):
    def __init__(self, patch_size=16, num_classes=8192, embed_dim=768, depth=6,
                 num_heads=12, mlp_ratio=4., qkv_bias=True, qk_scale=None, drop_rate=0., attn_drop_rate=0.,
                 drop_path_rate=0., norm_layer=None, init_values=None, num_patches=196, init_std=0.02, args=None, patch_shape=(14,14)):
        super().__init__()

        self.num_features = self.embed_dim = embed_dim
        self.patch_size = patch_size
        self.args = args

        dpr = [x.item() for x in torch.linspace(0, drop_path_rate, depth)]

        # context regressor
        self.regressor_blocks = nn.ModuleList([
            RegressorBlock(
                dim=embed_dim, num_heads=num_heads, mlp_ratio=mlp_ratio, qkv_bias=qkv_bias, qk_scale=qk_scale,
                drop=drop_rate, attn_drop=attn_drop_rate, drop_path=dpr[i], norm_layer=norm_layer,
                init_values=init_values)
            for i in range(depth)])

        dpr = [x.item() for x in torch.linspace(0, drop_path_rate, args.decoder_depth)]
        self.decoder_blocks = nn.ModuleList([
            Block(
                dim=embed_dim, num_heads=num_heads, mlp_ratio=mlp_ratio, qkv_bias=qkv_bias, qk_scale=qk_scale,
                drop=drop_rate, attn_drop=attn_drop_rate, drop_path=dpr[i], norm_layer=norm_layer,
                init_values=init_values)
            for i in range(args.decoder_depth)])

        self.norm = norm_layer(embed_dim)
        self.norm2 = norm_layer(embed_dim)
        self.head = nn.Linear(embed_dim, num_classes) if num_classes > 0 else nn.Identity()

        self.init_std = init_std

        # init the model
        trunc_normal_(self.head.weight, std=self.init_std)
        self.apply(self._init_weights)
        self.fix_init_weight()

    def fix_init_weight(self):
        def rescale(param, layer_id):
            param.div_(math.sqrt(2.0 * layer_id))

        for layer_id, layer in enumerate(self.regressor_blocks):
            rescale(layer.cross_attn.proj.weight.data, layer_id + 1)
            rescale(layer.mlp_cross.fc2.weight.data, layer_id + 1)

        for layer_id, layer in enumerate(self.decoder_blocks):
            rescale(layer.attn.proj.weight.data, layer_id + 1)
            rescale(layer.mlp.fc2.weight.data, layer_id + 1)

    def _init_weights(self, m):
        if isinstance(m, nn.Linear):
            trunc_normal_(m.weight, std=self.init_std)
            if isinstance(m, nn.Linear) and m.bias is not None:
                nn.init.constant_(m.bias, 0)
        elif isinstance(m, nn.LayerNorm):
            nn.init.constant_(m.bias, 0)
            nn.init.constant_(m.weight, 1.0)
        elif isinstance(m, nn.Conv2d):
            trunc_normal_(m.weight, std=self.init_std)
            if m.bias is not None:
                nn.init.constant_(m.bias, 0)

    @torch.jit.ignore
    def no_weight_decay(self):
        return {'pos_embed', 'cls_token'}

    def forward(self, x_masked, x_unmasked, pos_embed_masked, pos_embed_unmasked, bool_masked_pos):
        # latent contextual regressor
        for blk in self.regressor_blocks:
            x_masked = blk(x_masked, torch.cat([x_unmasked, x_masked], dim=1), pos_embed_masked, torch.cat([pos_embed_unmasked, pos_embed_masked], dim=1), bool_masked_pos)
        x_masked = self.norm(x_masked)

        x_masked = x_masked + pos_embed_masked  # add pos embed, like encoder
        for blk in self.decoder_blocks:
            x_masked = blk(x_masked)

        return x_masked


class VisionTransformerForMaskedImageModeling(nn.Module):
    def __init__(self, img_size=224, patch_size=16, in_chans=3, vocab_size=8192, embed_dim=768, depth=12,
                 num_heads=12, mlp_ratio=4., qkv_bias=True, qk_scale=None, drop_rate=0., attn_drop_rate=0.,
                 drop_path_rate=0., norm_layer=None, init_values=None, attn_head_dim=None,
                 use_abs_pos_emb=True, init_std=0.02, decoder_embed_dim=768, args=None, **kwargs):
        super().__init__()

        self.encoder = VisionTransformerEncoder(img_size=img_size, patch_size=patch_size, in_chans=in_chans,
                 vocab_size=vocab_size, embed_dim=embed_dim, depth=depth,
                 num_heads=num_heads, mlp_ratio=mlp_ratio, qkv_bias=qkv_bias, qk_scale=qk_scale,
                 drop_rate=drop_rate, attn_drop_rate=attn_drop_rate, drop_path_rate=drop_path_rate,
                 norm_layer=norm_layer, init_values=init_values, attn_head_dim=attn_head_dim,
                 use_abs_pos_emb=use_abs_pos_emb, init_std=init_std, args=args)

        # alignment constraint
        self.teacher = VisionTransformerEncoder(img_size=img_size, patch_size=patch_size, in_chans=in_chans,
                vocab_size=vocab_size, embed_dim=embed_dim, depth=depth,
                num_heads=num_heads, mlp_ratio=mlp_ratio, qkv_bias=qkv_bias, qk_scale=qk_scale,
                drop_rate=drop_rate, attn_drop_rate=attn_drop_rate, drop_path_rate=drop_path_rate,
                norm_layer=norm_layer, init_values=init_values, attn_head_dim=attn_head_dim,
                use_abs_pos_emb=use_abs_pos_emb, init_std=init_std, args=args)

        self.image_decoder = nn.Sequential(
                nn.Linear(embed_dim, embed_dim),
                nn.ReLU(),
                nn.Linear(embed_dim, embed_dim),
                nn.ReLU(),
                nn.Linear(embed_dim, embed_dim),
                nn.ReLU(),
                nn.Linear(embed_dim, embed_dim))

        self.init_std = init_std
        self.args = args
        self.num_patches = self.encoder.patch_embed.num_patches
        self.patch_embed = PatchEmbed(
                              img_size=img_size,
                              patch_size=patch_size,
                              in_chans=in_chans,
                              embed_dim=embed_dim)

        self.pretext_neck = VisionTransformerNeck(patch_size=patch_size, num_classes=768, embed_dim=decoder_embed_dim, depth=args.regressor_depth,
            num_heads=args.decoder_num_heads, mlp_ratio=mlp_ratio, qkv_bias=qkv_bias, qk_scale=qk_scale, drop_rate=drop_rate, attn_drop_rate=attn_drop_rate,
            drop_path_rate=drop_path_rate, norm_layer=norm_layer, init_values=args.decoder_layer_scale_init_value, num_patches=self.num_patches, init_std=init_std, args=args)


        # encoder to decoder projection, borrowed from mae.
        if decoder_embed_dim != embed_dim:
            self.encoder_to_decoder = nn.Linear(embed_dim, decoder_embed_dim, bias=True)
            self.encoder_to_decoder_norm = norm_layer(decoder_embed_dim)
        else:
            self.encoder_to_decoder = None

        self.mask_token = nn.Parameter(torch.zeros(1, 1, decoder_embed_dim))
        trunc_normal_(self.mask_token, std=self.init_std)

        ### whether to use 'rescale' to init the weight, borrowed from beit.
        if not args.fix_init_weight:
            self.apply(self._init_weights)
        self._init_teacher()


    def _init_teacher(self):
        # init the weights of teacher with those of backbone
        for param_encoder, param_teacher in zip(self.encoder.parameters(), self.teacher.parameters()):
            param_teacher.detach()
            param_teacher.data.copy_(param_encoder.data)
            param_teacher.requires_grad = False

    def momentum_update(self, base_momentum=0):
        """Momentum update of the teacher network."""
        for param_encoder, param_teacher in zip(self.encoder.parameters(),
                                                self.teacher.parameters()):
            param_teacher.data = param_teacher.data * base_momentum + \
                param_encoder.data * (1. - base_momentum)

    def _init_weights(self, m):
        if isinstance(m, nn.Linear):
            nn.init.xavier_uniform_(m.weight)
            if isinstance(m, nn.Linear) and m.bias is not None:
                nn.init.constant_(m.bias, 0)
        elif isinstance(m, nn.LayerNorm):
            nn.init.constant_(m.bias, 0)
            nn.init.constant_(m.weight, 1.0)

    '''
    Input shape:
        x: [bs, 3, 224, 224]
        bool_masked_pos: [bs, num_patch * num_patch]
    '''
    def forward(self, x, bool_masked_pos, return_all_tokens=None):

        '''
        Encoder
        Output shape:
            [bs, num_visible + 1, C]
        '''
        mask_pos = bool_masked_pos.bool()
        # mask_pos = torch.logical_not(bool_masked_pos)
        unmask_pos = torch.logical_not(mask_pos)
        x_unmasked = self.encoder(x, bool_masked_pos=bool_masked_pos) # encoded visible patches

        x_embed = self.patch_embed(x)

        # unmasked embeddings
        batch_size, seq_len, dim = x_embed.size()
        x_mask_org = x_embed[mask_pos].reshape(batch_size, -1, dim) # mask patches
        # x_mask_org = x_embed.masked_fill(bool_unmasked_pos_unsqueeze.bool(), 0.) # ver 2

        # x_mask_org = x[bool_masked_pos.numpy().astype(np.int32)] # ver1

        # encoder to decoder projection
        if self.encoder_to_decoder is not None:
            x_unmasked = self.encoder_to_decoder(x_unmasked)
            x_unmasked = self.encoder_to_decoder_norm(x_unmasked)

        '''
        Alignment constraint
        '''
        with torch.no_grad():
            latent_target = self.teacher(x, bool_masked_pos=(unmask_pos)) # since the modified encoder always takes mask pos and invert that mask to work on visible pos
            latent_target = latent_target[:, 1:, :] # remove class token
            if self.encoder_to_decoder is not None:
                latent_target = self.encoder_to_decoder_norm(self.encoder_to_decoder(latent_target.detach()))

            self.momentum_update(self.args.base_momentum)
        '''
        Latent contextual regressor and decoder
        '''
        b, num_visible_plus1, dim = x_unmasked.shape

        num_masked_patches = self.num_patches - (num_visible_plus1-1)

        # generate position embeddings.
        pos_embed = self.encoder.build_2d_sincos_position_embedding(dim, use_cls_token=True).expand(batch_size, self.num_patches+1, dim).to(self.args.device)

        # pos embed for masked patches
        pos_embed_masked = pos_embed[:,1:][mask_pos].reshape(batch_size, -1, dim)
        pos_embed_masked = torch.cat((pos_embed[:,:1], pos_embed_masked),dim=1)

        # pos embed for unmasked patches
        pos_embed_unmasked = pos_embed[:,1:][unmask_pos].reshape(batch_size, -1, dim)
        pos_embed_unmasked = torch.cat((pos_embed[:,:1], pos_embed_unmasked),dim=1)
        x_unmasked = x_unmasked + pos_embed_unmasked

        # masked embedding '''
        x_masked = self.mask_token.expand(batch_size, num_masked_patches+1, -1)

        # predict masked image
        x_masked = x_masked + pos_embed_masked
        x_masked = self.pretext_neck(x_masked, x_unmasked, pos_embed_masked, pos_embed_unmasked, bool_masked_pos)
        # remove class token
        x_masked = x_masked[:, 1:, :]

        x_masked_pred_decode = self.image_decoder(x_masked)

        return x_masked_pred_decode, x_mask_org, x_masked, latent_target

def _cfg(url='', **kwargs):
    return {
        'url': url,
        'num_classes': 1000, 'input_size': (3, 224, 224), 'pool_size': None,
        'crop_pct': .9, 'interpolation': 'bicubic',
        'mean': (0.5, 0.5, 0.5), 'std': (0.5, 0.5, 0.5),
        **kwargs
    }


def maskocr_encoder_pretrain(pretrained=False, **kwargs):
    model = VisionTransformerForMaskedImageModeling(
        patch_size = kwargs.patch_size, # 16
        embed_dim = kwargs.embed_dim, # 384
        depth = kwargs.depth, # 12
        num_heads = kwargs.num_heads, # 12
        mlp_ratio = kwargs.mlp_ratio, # 4
        qkv_bias = kwargs.qkv_bias, # True
        norm_layer = partial(nn.LayerNorm, eps=1e6),
        vocab_size = kwargs.vocab_size, # 8192
    )

    model.default_cfg = _cfg()

    if pretrained:
        checkpoint = torch.load(
            kwargs.init_ckpt, map_location='cpu'
        )
        model.load_state_dict(checkpoint['model'])
    return model


class CELayer(nn.Module):
    '''
    Cross-entropy over character positions.
        y_pred: [num_characters, bs, num_classes] (decoder output)
        y_true: [bs, num_characters], positions set to -100 are ignored
        label_length: [bs, 1] (optional) number of characters in each label; the loss then covers
                      the characters plus the first [EOS], as in the MaskOCR paper
    '''
    def __init__(self):
        super(CELayer, self).__init__()
        self.loss_fn = nn.CrossEntropyLoss(ignore_index=-100)

    def forward(self, y_true, y_pred, label_length=None):
        y_pred = y_pred.permute(1, 2, 0) # -> [bs, num_classes, num_characters]

        if label_length is not None:
            positions = torch.arange(y_true.size(1), device=y_true.device).unsqueeze(0)
            keep = positions <= label_length.reshape(-1, 1).to(y_true.device)
            y_true = y_true.masked_fill(~keep, -100)

        return self.loss_fn(y_pred, y_true)


def edit_distance(a, b):
    # Levenshtein distance: substitutions + deletions + insertions
    prev = list(range(len(b) + 1))
    for i, ca in enumerate(a, 1):
        cur = [i]
        for j, cb in enumerate(b, 1):
            cur.append(min(prev[j] + 1, cur[j - 1] + 1, prev[j - 1] + (ca != cb)))
        prev = cur
    return prev[-1]


def cer(preds, refs):
    # Character Error Rate = (S + D + I) / N over the whole set
    total_chars = sum(len(r) for r in refs)
    total_edits = sum(edit_distance(p, r) for p, r in zip(preds, refs))
    return total_edits / max(total_chars, 1)


def load_encoder_from_pretrain(model, path):
    # copy the encoder weights out of an encoder-pretraining checkpoint (model_*.pth / EncoderModel_*.pth)
    state_dict = torch.load(path, map_location='cpu')
    encoder_state = {k[len('encoder.'):]: v for k, v in state_dict.items() if k.startswith('encoder.')}
    model.encoder.load_state_dict(encoder_state)
    logger(f'Pretrained encoder loaded from {path}')


def load_matching_weights(model, path):
    # load every weight whose name and shape match (the vocabulary size can differ between stages)
    state_dict = torch.load(path, map_location='cpu')
    own_state = model.state_dict()
    matched = {k: v for k, v in state_dict.items() if k in own_state and own_state[k].shape == v.shape}
    model.load_state_dict(matched, strict=False)
    skipped = sorted(set(own_state) - set(matched))
    logger(f'Loaded {len(matched)}/{len(own_state)} weights from {path}' + (f' (not loaded: {skipped})' if skipped else ''))

class MaskOCRDecoder(nn.Module):
    def __init__(self, embed_dim=768, output_dim=500, num_heads=12, depth=4, args=None):
        super(MaskOCRDecoder, self).__init__()
        decoder_layer = nn.TransformerDecoderLayer(d_model=embed_dim, nhead=num_heads)
        self.decoder = nn.TransformerDecoder(decoder_layer, num_layers=depth)
        self.args = args
        self.embed_dim = embed_dim

        self.mask_token = 0.
        # classifier
        self.linear_classifier = nn.Linear(in_features=embed_dim, out_features=output_dim)

        # learnable character queries
        self.character_queries = self.get_character_queries(embed_dim)
        # Initialize character queries
        self.character_queries.weight.data.uniform_(-0.1, 0.1)

    def get_character_queries(self, embed_dim):
            return nn.Embedding(self.args.max_text_length, embed_dim)

    def forward(self, encoded_patches, bool_masked_pos=None):
        T, B, C = encoded_patches.shape
        char_queries = self.character_queries(torch.arange(self.args.max_text_length).unsqueeze(0).repeat(B, 1).to(self.args.device))
        # masked character queries if decoder is in pretraining phase
        if bool_masked_pos != None:
            # 1 - bool_masked_pos | bool_masked_pos
            B, T = bool_masked_pos.shape
            # loop per item in batch:
            count = 0
            temp = None
            for item in bool_masked_pos:
                item = item[item != 999].reshape(-1)
                if count == 0:
                    temp = torch.nn.functional.pad(item, (0, self.args.max_text_length - len(item)), value=self.mask_token)
                else:
                    temp = torch.cat([temp, torch.nn.functional.pad(item, (0, self.args.max_text_length - len(item)), value=self.mask_token)], dim=0)
                count = 1
            bool_masked_pos = temp.reshape(B, -1).bool() # since 0 is visible and 1 is masked, -> must revert this to times with the char queries to mask the char queries
            bool_masked_pos = torch.logical_not(bool_masked_pos)

            char_queries *= bool_masked_pos.unsqueeze(2).float()


        char_queries = char_queries.permute(1, 0, 2) # [bs, max_text_length, C] -> [max_text_length, bs, C]
        encoded_patches = self.decoder(char_queries, encoded_patches)
        encoded_patches = self.linear_classifier(encoded_patches)
        return encoded_patches


class MaskOCR(nn.Module):
    def __init__(self, encoder_args, decoder_args, args):
        super(MaskOCR, self).__init__()
        self.e_args = encoder_args
        self.d_args = decoder_args
        self.args = args

        self.mode = self.args.mode # mode = 'e_pretrain', 'd_pretrain', 'train', 'test'


        # encoder
        self.encoder = VisionTransformerEncoder(img_size=encoder_args.img_size,
                                                patch_size=encoder_args.patch_size,
                                                in_chans=encoder_args.in_chans,
                                                vocab_size=encoder_args.vocab_size,
                                                embed_dim=encoder_args.embed_dim,
                                                depth=encoder_args.depth,
                                                num_heads=encoder_args.num_heads,
                                                mlp_ratio=encoder_args.mlp_ratio,
                                                qkv_bias=encoder_args.qkv_bias,
                                                qk_scale=encoder_args.qk_scale,
                                                drop_rate=encoder_args.drop_rate,
                                                attn_drop_rate=encoder_args.attn_drop_rate,
                                                drop_path_rate=encoder_args.drop_path_rate,
                                                norm_layer=encoder_args.norm_layer,
                                                init_values=encoder_args.init_values,
                                                attn_head_dim=encoder_args.attn_head_dim,
                                                use_abs_pos_emb=encoder_args.use_abs_pos_emb,
                                                init_std=encoder_args.init_std,
                                                args=encoder_args.args)
        # decoder
        self.decoder = MaskOCRDecoder(embed_dim=decoder_args.embed_dim,
                                      output_dim=decoder_args.output_dim,
                                      num_heads=decoder_args.num_heads,
                                      depth=decoder_args.depth,
                                      args=self.d_args)

    def set_d_pretrain(self):
        self.encoder.eval()
        self.decoder.train(True)


    def forward(self, patches, bool_masked_pos_extend, bool_masked_pos=None):
        if self.args.mode == 'd_pretrain':
            # decoder pretraining: the encoder is frozen
            with torch.no_grad():
                patches = self.encoder(patches, bool_masked_pos=bool_masked_pos_extend).permute(1, 0, 2)
            patches = self.decoder(patches, bool_masked_pos)
        else:
            # main training: encoder and decoder are trained together
            patches = self.encoder(patches, bool_masked_pos=bool_masked_pos_extend).permute(1, 0, 2)
            patches = self.decoder(patches)
        return patches

    def greedy_decode(self, out, vocab):
        '''
        out: [num_characters, bs, num_classes] -> list of predicted strings.
        Takes the most likely class at each character position and stops at [EOS] ('Ë').
        '''
        eos = len(vocab) - 1
        texts = []
        for item in out.argmax(-1).permute(1, 0).tolist():
            text = ''
            for idx in item:
                if idx >= eos:
                    break
                text += vocab[idx]
            texts.append(text)
        return texts



## MaskOCR Encoder
This part is for pretraining encoder

### Pretraining pipeline for Encoder

In [ ]:
from datetime import datetime
from torch.utils.tensorboard import SummaryWriter
from tqdm import tqdm


class EncoderPretrainer():
    def __init__(self, kwargs):
        self.kwargs = kwargs

        # build the encoder with the configured size, so it can be loaded into MaskOCR later
        self.model = VisionTransformerForMaskedImageModeling(img_size=kwargs.encoder.img_size,
                                                             patch_size=kwargs.encoder.patch_size,
                                                             in_chans=kwargs.encoder.in_chans,
                                                             embed_dim=kwargs.encoder.embed_dim,
                                                             depth=kwargs.encoder.depth,
                                                             num_heads=kwargs.encoder.num_heads,
                                                             mlp_ratio=kwargs.encoder.mlp_ratio,
                                                             qkv_bias=kwargs.encoder.qkv_bias,
                                                             decoder_embed_dim=kwargs.encoder.decoder_embed_dim,
                                                             norm_layer=kwargs.encoder.norm_layer,
                                                             args=kwargs.encoder,
                                                             init_values=kwargs.encoder.layer_scale_init_value).cpu()
        self.lr = kwargs.encoder.lr
        self.ld = kwargs.encoder.ld
        self.optimizer = self.get_optimizer(kwargs.encoder.optimizer)

    def get_optimizer(self, name):
        if name.lower() == 'sgd':
            return torch.optim.SGD(self.model.parameters(), lr=self.lr, momentum=0.9)
        elif name.lower() == 'adamw':
            return torch.optim.AdamW(self.model.parameters(), lr=self.lr)
        elif name.lower() == 'adam':
            return torch.optim.Adam(self.model.parameters(), lr=self.lr)
        else:
            raise Exception(f'The optimizer {name} was not inplemented.')

    def train_one_epoch(self, epoch_index, tb_writer, dataloader):
        running_loss = 0.
        epoch_loss = 0.
        running_latent_loss = 0.
        running_recon_loss = 0.


        # Here, we use enumerate(training_loader) instead of
        # iter(training_loader) so that we can track the batch
        # index and do some intra-epoch reporting
        batch_iter = tqdm(dataloader, desc='Batch')
        for i, data in enumerate(batch_iter):
            # Every data instance is a tuple of patches, visual_tokens and masks
            patches, visual_tokens, masks = data

            patches = patches.to(self.kwargs.device)
            visual_tokens = visual_tokens.to(self.kwargs.device)
            masks = masks.to(self.kwargs.device)

            # Zero your gradients for every batch!
            self.optimizer.zero_grad()

            # Make predictions for this batch
            pred_mask_img, true_mask_img, pred_tokens, true_tokens = self.model(patches, masks)
            latent_loss = F.mse_loss(true_tokens, pred_tokens)
            recon_loss = F.mse_loss(true_mask_img, pred_mask_img)

            # Compute the loss and its gradients
            loss = self.ld*latent_loss + recon_loss
            loss.backward()

            # Adjust learning weights
            self.optimizer.step()

            # Gather data and report
            batch_iter.set_postfix({'loss': loss.item()})
            running_loss += loss.item()
            epoch_loss += loss.item()
            running_latent_loss += latent_loss.item()
            running_recon_loss += recon_loss.item()

            log_step = 100
            if i % log_step == log_step-1:
                print('  batch: {}\t-\tloss: {}'.format(i + 1, running_loss / log_step))
                tb_x = epoch_index * len(dataloader) + i + 1

                tb_writer.add_scalar('Loss/latent_loss', running_latent_loss/log_step, tb_x)
                tb_writer.add_scalar('Loss/recon_loss', running_recon_loss/log_step, tb_x)
                tb_writer.add_scalar('Loss/train', running_loss / log_step, tb_x)
                running_loss = 0.
                running_latent_loss = 0.
                running_recon_loss = 0.

        return epoch_loss / len(dataloader) # mean loss over the epoch

    def pretrain(self):
        # Initializing in a separate cell so we can easily add more epochs to the same run
        timestamp = datetime.now().strftime('%Y%m%d_%H%M%S')
        writer = SummaryWriter('runs/encoder_trainer_{}'.format(timestamp))
        epoch_number = 0

        EPOCHS = self.kwargs.encoder.epochs

        best_vloss = 1_000_000.

        # load dataset
        print('[INFO] Loading dataset...')
        dataset = OCR_Dataset(self.kwargs.encoder)
        train_size = int(self.kwargs.encoder.train_ratio * len(dataset))
        test_size = int((len(dataset) - train_size)/2)
        val_size = len(dataset) - train_size - test_size

        # split dataset to train and test
        train_dataset, test_dataset, val_dataset = random_split(dataset, [train_size, test_size, val_size])

        train_loader = DataLoader(train_dataset, batch_size=self.kwargs.encoder.batch_size, shuffle=True, num_workers=self.kwargs.encoder.n_workers, worker_init_fn=seed_worker)
        test_loader = DataLoader(test_dataset, batch_size=self.kwargs.encoder.batch_size, shuffle=False, num_workers=self.kwargs.encoder.n_workers, worker_init_fn=seed_worker)
        val_loader = DataLoader(val_dataset, batch_size=self.kwargs.encoder.batch_size, shuffle=False, num_workers=self.kwargs.encoder.n_workers, worker_init_fn=seed_worker)
        print('[INFO] Done, continue...')

        # load encoder to cuda
        self.model.to(self.kwargs.encoder.device)
        print(f'[INFO] Loaded model to {self.kwargs.encoder.device}')


        # start training model
        print('[INFO] Start training model...')
        for epoch in range(EPOCHS):
            print('EPOCH {}:'.format(epoch_number + 1))

            # Make sure gradient tracking is on, and do a pass over the data
            self.model.train(True)
            avg_loss = self.train_one_epoch(epoch_number, writer, train_loader)


            running_vloss = 0.0
            # Set the model to evaluation mode, disabling dropout and using population
            # statistics for batch normalization.
            self.model.eval()

            # Disable gradient computation and reduce memory consumption.
            print('[INFO] Validating model...')
            with torch.no_grad():
                batch_iter = tqdm(val_loader, desc='Batch')
                for i, vdata in enumerate(val_loader):
                    patches, visual_tokens, masks = vdata
                    patches = patches.to(self.kwargs.encoder.device)
                    visual_tokens = visual_tokens.to(self.kwargs.encoder.device)
                    masks = masks.to(self.kwargs.encoder.device)
                    pred_mask_img, true_mask_img, pred_tokens, true_tokens = self.model(patches, masks)
                    latent_loss = F.mse_loss(true_tokens, pred_tokens)
                    recon_loss = F.mse_loss(true_mask_img, pred_mask_img)

                    # Compute the loss and its gradients
                    running_vloss += self.ld*latent_loss + recon_loss
                    # vinputs, vlabels = vdata
                    # voutputs = self.model(vinputs)
                    # vloss = F.mse_loss(voutputs, vlabels)
                    # running_vloss += vloss

            avg_vloss = running_vloss / (i + 1)
            print('\tLOSS train {} valid {}'.format(avg_loss, avg_vloss))

            # Log the running loss averaged per batch
            # for both training and validation
            writer.add_scalars('Training vs. Validation Loss',
                            { 'Training' : avg_loss, 'Validation' : avg_vloss },
                            epoch_number + 1)
            writer.flush()

            # Track best performance, and save the model's state
            if avg_vloss < best_vloss:
                best_vloss = avg_vloss
                model_path = 'EncoderModel_{}_{}.pth'.format(timestamp, epoch_number)
                torch.save(self.model.state_dict(), model_path)

            epoch_number += 1

### Training

In [ ]:

trainer = EncoderPretrainer(
              kwargs=Obj({'encoder':{
                    "checkpoint_path": "saved_models/MaskOCREncoder/",
                    'dataset': {
                        'name': ['iam', 'vi'],
                        'iam_path': '/content/data',
                        'vi_path': '/content/data/vi',
                    },
                    "mode": "e_pretrain",
                    "img_size": 224,
                    "patch_size": 16,
                    "in_chans": 3,
                    "embed_dim": 768,
                    "depth": 6,
                    "num_heads": 6,
                    "mlp_ratio": 4.,
                    "qkv_bias": True,
                    "qk_scale": None,
                    "drop_rate": 0.,
                    "attn_drop_rate": 0.,
                    "drop_path_rate": 0.,
                    "norm_layer": partial(nn.LayerNorm, eps=1e-6),
                    "init_values": None,
                    "attn_head_dim": None,
                    "use_abs_pos_emb": True,
                    "init_std": 0.02,
                    "decoder_embed_dim": 768,
                    "pretrain_ld": 0.05, # as in MaskOCR paper

                    # hyper params
                    "optimizer": "adamw",
                    "layer_scale_init_value": 0.1,
                    "regressor_depth": 4, # depth of the regressor
                    "decoder_depth": 4, # depth of the image decoder
                    "decoder_embed_dim": 768, # dimensionality of embeddings for the image decoder
                    "decoder_num_heads": 4, # Number of heads for the image decoder
                    "decoder_layer_scale_init_value": 0.1, # decoder layer scale init value

                    # alignment constraint
                    "base_momentum": 0, # ema weight for the dual path network
                    "fix_init_weight": False, # if true, the fix_init_weight() func will be activated

                    # data param
                    "lr": 1.5e-4,
                    "batch_size": 80,
                    "epochs": 5,
                    "ratio_mask_patches": 0.4,
                    "device": "cuda",
                    "ld": 0.05,
                    "input_size": 224,
                    "color_jitter": 0,
                    "train_ratio": 0.8,
                    "n_workers": 2, # 2 * num gpu
                    "imagenet_default_mean_and_std": True,
                    },
                "ratio_mask_patches": 0.4,
                "device": "cuda",
                "mode": "e_pretrain",
                "ld": 0.05 # as in MaskOCR paper
            }))

# pretrain encoder
trainer.pretrain()

In [ ]:
# plot training logs
%load_ext tensorboard
%tensorboard --logdir runs

## MaskOCR Decoder
This part is for pretraining decoder

### Pretrain decoder

#### Pretraining pipeline for decoder

In [ ]:
class DecoderPretrainer():
    def __init__(self, args):
        self.args = args

        self.dataset = None
        self.train_loader = None
        self.test_loader = None
        self.val_loader = None
        self.load_dataset()

        self.ce_layer = CELayer()

        # set decoder output dim equal to vocab's length
        if args.decoder.output_dim == None:
            args.decoder.output_dim = len(self.dataset.vocab) + 1

        if args.decoder.max_text_length == None:
            args.decoder.max_text_length = self.dataset.max_text_length

        if args.encoder.vocab_size == None:
            args.encoder.vocab_size = len(self.dataset.vocab)

        self.model = MaskOCR(encoder_args=args.encoder, decoder_args=args.decoder, args=args).cpu()
        # check pretrain encoder model
        if self.args.pretrain_encoder_path != None:
            self.load_encoder_weight(self.args.pretrain_encoder_path)

        self.lr = args.lr

        self.optimizer = self.get_optimizer(args.optimizer)

    def get_optimizer(self, name):
        if name.lower() == 'sgd':
            return torch.optim.SGD(self.model.parameters(), lr=self.lr, momentum=0.9)
        elif name.lower() == 'adamw':
            return torch.optim.AdamW(self.model.parameters(), lr=self.lr)
        elif name.lower() == 'adam':
            return torch.optim.Adam(self.model.parameters(), lr=self.lr)
        else:
            raise Exception(f'The optimizer {name} was not inplemented.')

    def load_dataset(self):
        # load dataset
        logger('Loading dataset...')
        self.dataset = OCR_Dataset(self.args)
        train_size = int(self.args.train_ratio * len(self.dataset))
        test_size = int((len(self.dataset) - train_size)/2)
        val_size = len(self.dataset) - train_size - test_size

        # split dataset to train and test
        train_dataset, test_dataset, val_dataset = random_split(self.dataset, [train_size, test_size, val_size])

        self.train_loader = DataLoader(train_dataset, batch_size=self.args.batch_size, shuffle=True, num_workers=self.args.n_workers, worker_init_fn=seed_worker)
        self.test_loader = DataLoader(test_dataset, batch_size=self.args.batch_size, shuffle=False, num_workers=self.args.n_workers, worker_init_fn=seed_worker)
        self.val_loader = DataLoader(val_dataset, batch_size=self.args.batch_size, shuffle=False, num_workers=self.args.n_workers, worker_init_fn=seed_worker)
        logger('Dataset loaded.')

    def load_encoder_weight(self, path):
        load_encoder_from_pretrain(self.model, path)

    def masked_targets(self, mask_labels):
        # loss only on the masked characters: every other position holds the padding token -> ignored
        return mask_labels.masked_fill(mask_labels == self.dataset.padding_token, -100)

    def train_one_epoch(self, epoch_index, tb_writer):
        running_loss = 0.
        epoch_loss = 0.

        # Here, we use enumerate(training_loader) instead of
        # iter(training_loader) so that we can track the batch
        # index and do some intra-epoch reporting
        batch_iter = tqdm(self.train_loader, desc='Batch')
        for i, data in enumerate(batch_iter):
            # Every data instance is a tuple of patches, visual_tokens and masks
            patches, text_masks, masks_extend, labels, mask_labels, lengths = data

            patches = patches.to(self.args.device)
            text_masks = text_masks.to(self.args.device)
            masks_extend = masks_extend.to(self.args.device)
            labels = labels.to(self.args.device)
            mask_labels = mask_labels.to(self.args.device)

            # Zero your gradients for every batch!
            self.optimizer.zero_grad()

            # Make predictions for this batch
            outputs = self.model(patches, masks_extend, text_masks)

            # Compute the loss and its gradients
            loss = self.ce_layer(self.masked_targets(mask_labels), outputs)
            loss.backward()

            # Adjust learning weights
            self.optimizer.step()

            # Gather data and report (first item of the batch)
            batch_iter.set_postfix({'loss': loss.item(),
                                    'true': self.dataset.decode_label(labels[0]),
                                    'pred': self.model.greedy_decode(outputs[:, :1, :], self.dataset.vocab)[0]})
            running_loss += loss.item()
            epoch_loss += loss.item()
            log_step = 100
            if i % log_step == log_step-1:
                print('  batch: {}\t-\tloss: {}'.format(i + 1, running_loss / log_step))
                tb_x = epoch_index * len(self.train_loader) + i + 1
                tb_writer.add_scalar('Loss/train', running_loss / log_step, tb_x)
                running_loss = 0.

        return epoch_loss / len(self.train_loader) # mean loss over the epoch

    def train(self):
        # Initializing in a separate cell so we can easily add more epochs to the same run
        timestamp = datetime.now().strftime('%Y%m%d_%H%M%S')
        writer = SummaryWriter('runs/decoder_trainer_{}'.format(timestamp))
        epoch_number = 0

        EPOCHS = self.args.epochs

        best_vloss = 1_000_000.

        # load encoder to cuda
        self.model.to(self.args.device)
        print(f'[INFO] Loaded model to {self.args.device}')


        # start training model
        print('[INFO] Start training MaskOCR decoder...')

        for epoch in range(EPOCHS):
            print('EPOCH {}:'.format(epoch_number + 1))

            self.model.set_d_pretrain() # set training mode (encoder frozen)

            # Make sure gradient tracking is on, and do a pass over the data
            avg_loss = self.train_one_epoch(epoch_number, writer)

            running_vloss = 0.0
            # Set the model to evaluation mode, disabling dropout and using population
            # statistics for batch normalization.
            self.model.eval()

            # Disable gradient computation and reduce memory consumption.
            print('[INFO] Validating model...')
            with torch.no_grad():
                batch_iter = tqdm(self.val_loader, desc='Batch')
                for i, vdata in enumerate(self.val_loader):
                    patches, masks, masks_extend, labels, mask_labels, lengths = vdata

                    labels = labels.to(self.args.device)
                    mask_labels = mask_labels.to(self.args.device)
                    patches = patches.to(self.args.device)
                    masks = masks.to(self.args.device)
                    masks_extend = masks_extend.to(self.args.device)

                    outputs = self.model(patches, masks_extend, masks)

                    # Compute the loss and its gradients
                    running_vloss += self.ce_layer(self.masked_targets(mask_labels), outputs).item()

            avg_vloss = running_vloss / (i + 1)
            print('\tLOSS train {} valid {}'.format(avg_loss, avg_vloss))

            # Log the running loss averaged per batch
            # for both training and validation
            writer.add_scalars('Training vs. Validation Loss',
                            { 'Training' : avg_loss, 'Validation' : avg_vloss },
                            epoch_number + 1)
            writer.flush()

            # Track best performance, and save the model's state
            if avg_vloss < best_vloss:
                best_vloss = avg_vloss
                model_path = 'DecoderModel_{}_{}.pth'.format(timestamp, epoch_number)
                torch.save(self.model.state_dict(), model_path)

            epoch_number += 1


#### Training

In [ ]:
batch_size = 256 #@param{type: "number"}
epochs = 5 #@param{type: "number"}
num_heads = 6 #@param{type: "number"}
depth = 6 #@param{type: "number"}
pretrain_encoder_path = None #@param {type: "raw"}  e.g. '/content/EncoderModel_<timestamp>_<epoch>.pth'


trainer = DecoderPretrainer(
              args=Obj({
                'encoder': {
                    # hyper params
                    "optimizer": "adamw",
                    "layer_scale_init_value": 0.1,
                    "regressor_depth": 4, # depth of the regressor
                    "decoder_depth": 4, # depth of the image decoder
                    "decoder_embed_dim": 768, # dimensionality of embeddings for the image decoder
                    "decoder_num_heads": 6, # Number of heads for the image decoder
                    "decoder_layer_scale_init_value": 0.1, # decoder layer scale init value
                    # alignment constraint
                    "base_momentum": 0, # ema weight for the dual path network
                    "fix_init_weight": False, # if true, the fix_init_weight() func will be activated
                    'imagenet_default_mean_and_std': True,

                    'img_size': 224,
                    'patch_size': 16,
                    'in_chans': 3,
                    'vocab_size': None,
                    'embed_dim': 768,
                    'depth': depth,
                    'num_heads': num_heads,
                    'mlp_ratio': 4.,
                    'qkv_bias': True,
                    'qk_scale': None,
                    'drop_rate': 0.,
                    'attn_drop_rate': 0.,
                    'drop_path_rate': 0.,
                    'norm_layer': partial(nn.LayerNorm, eps=1e-6),
                    'init_values': 0.4,
                    'attn_head_dim': None,
                    'use_abs_pos_emb': True,
                    'init_std': 0.02,
                    'args': {
                    }
                    },
                'decoder': {
                    'device': 'cuda',
                    'embed_dim': 768,
                    'batch_size': batch_size,
                    'output_dim': None,
                    'max_text_length': None,
                    'num_heads': num_heads,
                    'depth': depth,
                    'num_patches': 16,
                },
                # dataloader hyperparams
                'mode': 'd_pretrain',
                'patch_size': 16,
                'train_ratio': 0.85,
                'input_size': 224,
                'ratio_mask_patches': 0.5,
                'imagenet_default_mean_and_std': True,
                'dataset': {
                    'name': ['iam', 'vi'],
                    'iam_path': '/content/data',
                    'vi_path': '/content/data/vi',
                },
                'color_jitter': 0,

                # model hyperparams
                'pretrain_encoder_path': pretrain_encoder_path, # checkpoint from encoder pretraining
                'pretrain_model_path': None, # not used in this stage
                'lr': 1.5e-4,
                'optimizer': 'adamw',
                'batch_size': 768,
                'n_workers': 2, # 2 * num gpu
                'device': 'cuda',
                'epochs': epochs,
            }))

# pretrain decoder
trainer.train()

## Train MaskOCR

### Training pipeline for MaskOCR

In [ ]:

class MaskOCR_Trainer():
    def __init__(self, kwargs):
        self.kwargs = kwargs

        self.dataset = None
        self.train_loader = None
        self.test_loader = None
        self.val_loader = None
        self.load_dataset()

        # set decoder output dim equal to vocab's length
        if kwargs.decoder.output_dim == None:
            kwargs.decoder.output_dim = len(self.dataset.vocab) + 1

        if kwargs.decoder.max_text_length == None:
            kwargs.decoder.max_text_length = self.dataset.max_text_length

        if kwargs.encoder.vocab_size == None:
            kwargs.encoder.vocab_size = len(self.dataset.vocab)

        self.model = MaskOCR(encoder_args=kwargs.encoder, decoder_args=kwargs.decoder, args=kwargs).cpu()

        # start from the pretraining stages
        if kwargs.pretrain_model_path != None:
            # checkpoint from decoder pretraining: holds both the (pretrained) encoder and the decoder
            load_matching_weights(self.model, kwargs.pretrain_model_path)
        elif kwargs.pretrain_encoder_path != None:
            load_encoder_from_pretrain(self.model, kwargs.pretrain_encoder_path)
        else:
            logger('No pretrained weights given: training from scratch.', type='WARNING')

        self.ce_layer = CELayer()
        self.lr = kwargs.lr
        self.optimizer = self.get_optimizer(kwargs.optimizer)

    def load_dataset(self):
        # load dataset
        logger('Loading dataset...')
        self.dataset = OCR_Dataset(self.kwargs)
        train_size = int(self.kwargs.train_ratio * len(self.dataset))
        test_size = int((len(self.dataset) - train_size)/2)
        val_size = len(self.dataset) - train_size - test_size

        # split dataset to train and test
        train_dataset, test_dataset, val_dataset = random_split(self.dataset, [train_size, test_size, val_size])

        self.train_loader = DataLoader(train_dataset, batch_size=self.kwargs.batch_size, shuffle=True, num_workers=self.kwargs.n_workers, worker_init_fn=seed_worker)
        self.test_loader = DataLoader(test_dataset, batch_size=self.kwargs.batch_size, shuffle=False, num_workers=self.kwargs.n_workers, worker_init_fn=seed_worker)
        self.val_loader = DataLoader(val_dataset, batch_size=self.kwargs.batch_size, shuffle=False, num_workers=self.kwargs.n_workers, worker_init_fn=seed_worker)
        logger('Dataset loaded.')

    def get_optimizer(self, name):
        if name.lower() == 'sgd':
            return torch.optim.SGD(self.model.parameters(), lr=self.lr, momentum=0.9)
        elif name.lower() == 'adamw':
            return torch.optim.AdamW(self.model.parameters(), lr=self.lr)
        elif name.lower() == 'adam':
            return torch.optim.Adam(self.model.parameters(), lr=self.lr)
        else:
            raise Exception(f'The optimizer {name} was not inplemented.')

    def train_one_epoch(self, epoch_index, tb_writer):
        running_loss = 0.
        epoch_loss = 0.

        # Here, we use enumerate(training_loader) instead of
        # iter(training_loader) so that we can track the batch
        # index and do some intra-epoch reporting
        batch_iter = tqdm(self.train_loader, desc='Batch')
        for i, data in enumerate(batch_iter):
            # Every data instance is a tuple of patches, visual_tokens, masks, labels and label lengths
            patches, _, masks, labels, lengths = data

            patches = patches.to(self.kwargs.device)
            masks = masks.to(self.kwargs.device)
            labels = labels.to(self.kwargs.device)

            # Zero your gradients for every batch!
            self.optimizer.zero_grad()

            # Make predictions for this batch
            outputs = self.model(patches, masks)

            # Compute the loss and its gradients
            loss = self.ce_layer(labels, outputs, lengths)
            loss.backward()

            # Adjust learning weights
            self.optimizer.step()

            # Gather data and report
            batch_iter.set_postfix({'loss': loss.item(),
                                    'true': self.dataset.decode_label(labels[0]),
                                    'pred': self.model.greedy_decode(outputs[:, :1, :], self.dataset.vocab)[0]})
            running_loss += loss.item()
            epoch_loss += loss.item()
            log_step = 100
            if i % log_step == log_step-1:
                print('  batch: {}\t-\tloss: {}'.format(i + 1, running_loss / log_step))
                tb_x = epoch_index * len(self.train_loader) + i + 1
                tb_writer.add_scalar('Loss/train', running_loss / log_step, tb_x)
                running_loss = 0.

        return epoch_loss / len(self.train_loader) # mean loss over the epoch

    def evaluate(self, dataloader):
        # returns (mean loss, CER) on a dataloader
        self.model.eval()
        total_loss = 0.
        preds, refs = [], []
        with torch.no_grad():
            for patches, _, masks, labels, lengths in tqdm(dataloader, desc='Evaluate'):
                patches = patches.to(self.kwargs.device)
                masks = masks.to(self.kwargs.device)
                labels = labels.to(self.kwargs.device)

                outputs = self.model(patches, masks)
                total_loss += self.ce_layer(labels, outputs, lengths).item()

                preds += self.model.greedy_decode(outputs, self.dataset.vocab)
                refs += [self.dataset.decode_label(label) for label in labels.cpu()]
        return total_loss / len(dataloader), cer(preds, refs)

    def train(self):
        # Initializing in a separate cell so we can easily add more epochs to the same run
        timestamp = datetime.now().strftime('%Y%m%d_%H%M%S')
        writer = SummaryWriter('runs/fashion_trainer_{}'.format(timestamp))
        epoch_number = 0

        EPOCHS = self.kwargs.epochs

        best_vloss = 1_000_000.

        # load encoder to cuda
        self.model.to(self.kwargs.device)
        print(f'[INFO] Loaded model to {self.kwargs.device}')


        # start training model
        print('[INFO] Start training model...')
        for epoch in range(EPOCHS):
            print('EPOCH {}:'.format(epoch_number + 1))

            # Make sure gradient tracking is on, and do a pass over the data
            self.model.train(True)
            avg_loss = self.train_one_epoch(epoch_number, writer)

            # Validation loss and character error rate
            print('[INFO] Validating model...')
            avg_vloss, val_cer = self.evaluate(self.val_loader)
            print('\tLOSS train {} valid {}\tCER valid {:.4f}'.format(avg_loss, avg_vloss, val_cer))
            writer.add_scalar('CER/validation', val_cer, epoch_number + 1)

            # Log the running loss averaged per batch
            # for both training and validation
            writer.add_scalars('Training vs. Validation Loss',
                            { 'Training' : avg_loss, 'Validation' : avg_vloss },
                            epoch_number + 1)
            writer.flush()

            # Track best performance, and save the model's state
            if avg_vloss < best_vloss:
                best_vloss = avg_vloss
                model_path = 'MaskOCR_{}_{}.pth'.format(timestamp, epoch_number)
                torch.save(self.model.state_dict(), model_path)
                best_model_path = model_path

            epoch_number += 1

        # final score on the held-out test split, using the best checkpoint
        self.model.load_state_dict(torch.load(best_model_path, map_location=self.kwargs.device))
        test_loss, test_cer = self.evaluate(self.test_loader)
        print(f'[INFO] Best model: {best_model_path}\tTEST loss {test_loss:.4f}\tCER {test_cer:.4f}')
        return test_cer

### Training

In [ ]:
batch_size = 768 #@param{type: "number"}
epochs = 20 #@param{type: "number"}
num_heads = 6 #@param{type: "number"}
depth = 6 #@param{type: "number"}
pretrain_encoder_path = None #@param {type: "raw"}  e.g. '/content/EncoderModel_<timestamp>_<epoch>.pth'
pretrain_model_path = None #@param {type: "raw"}  e.g. '/content/DecoderModel_<timestamp>_<epoch>.pth' (used first if set)

trainer = MaskOCR_Trainer(
              kwargs=Obj({
                'encoder': {
                    'img_size': 224,
                    'patch_size': 16,
                    'in_chans': 3,
                    'vocab_size': None,
                    'embed_dim': 768,
                    'depth': depth,
                    'num_heads': num_heads,
                    'mlp_ratio': 4.,
                    'qkv_bias': True,
                    'qk_scale': None,
                    'drop_rate': 0.,
                    'attn_drop_rate': 0.,
                    'drop_path_rate': 0.,
                    'norm_layer': partial(nn.LayerNorm, eps=1e-6),
                    'init_values': 0.5,
                    'attn_head_dim': None,
                    'use_abs_pos_emb': True,
                    'init_std': 0.02,
                    'args': {

                    }
                    },
                'decoder': {
                    'device': 'cuda',
                    'embed_dim': 768,
                    'batch_size': 32,
                    'output_dim': None,
                    'max_text_length': None,
                    'num_heads': num_heads,
                    'depth': depth,
                    'num_patches': 16,
                },
                # dataloader hyperparams
                'mode': 'train',
                'patch_size': 16,
                'train_ratio': 0.85,
                'input_size': 224,
                'ratio_mask_patches': 0.,
                'imagenet_default_mean_and_std': True,
                'dataset': {
                    'name': ['iam', 'vi'],
                    'iam_path': '/content/data',
                    'vi_path': '/content/data/vi',
                },
                'color_jitter': 0,

                # model hyperparams
                'pretrain_encoder_path': pretrain_encoder_path,
                'pretrain_model_path': pretrain_model_path,
                'lr': 1.5e-4,
                'optimizer': 'adamw',
                'batch_size': batch_size,
                'n_workers': 2, # 2 * num gpu
                'device': 'cuda',
                'epochs': epochs,
            }))

# train MaskOCR and report CER on the test split
trainer.train()